In [1]:
from secret_envs_wrapper import SecretEnv3
import sys
sys.path.append("..")  # Va chercher un dossier au-dessus
from monte_carlo_methods import on_policy_first_visit_mc_control
from monte_carlo_methods import off_policy_mc_control
#from monte_carlo_methods import mc_es_control

In [2]:
import random
from collections import defaultdict

def generate_episode(env, policy, max_steps=200):
    episode = []
    env.reset()
    state = env.state_id()
    previous_score = env.score()
    done = env.is_game_over()

    for _ in range(max_steps):
        if done:
            break

        valid_actions = env.available_actions()
        if len(valid_actions) == 0:
            print(f"[WARNING] No valid actions for state: {state}")
            break

        # Choix de l'action
        if state in policy and policy[state]:
            filtered_actions = [a for a in valid_actions if a in policy[state]]
            if filtered_actions:
                probs = [policy[state][a] for a in filtered_actions]
                action = random.choices(filtered_actions, weights=probs, k=1)[0]
            else:
                action = random.choice(valid_actions)
        else:
            action = random.choice(valid_actions)

        # Vérification stricte
        if action not in valid_actions:
            print(f"[WARNING] Action {action} non autorisée à l’état {state}")
            break

        try:
            env.step(action)
        except Exception as e:
            print(f"[ERROR] Exception pendant step : {e}")
            break

        next_state = env.state_id()
        new_score = env.score()
        reward = new_score - previous_score
        previous_score = new_score
        done = env.is_game_over()

        episode.append((state, action, reward))
        state = next_state

    return episode




def on_policy_first_visit_mc_control(env, epsilon, num_episodes):
    Q = defaultdict(lambda: defaultdict(float))
    Returns = defaultdict(list)
    policy = defaultdict(lambda: defaultdict(float))
    gamma = 0.99

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)
        G = 0

        for i in reversed(range(len(episode))):
            state, action, reward = episode[i]
            G = gamma * G + reward

            if not any(s == state and a == action for s, a, _ in episode[:i]):
                Returns[(state, action)].append(G)
                Q[state][action] = sum(Returns[(state, action)]) / len(Returns[(state, action)])

                if not policy[state]:
                    valid_actions = list(env.available_actions())
                    prob = 1.0 / len(valid_actions)
                    for a in valid_actions:
                        policy[state][a] = prob

                max_action = max(Q[state].items(), key=lambda x: x[1])[0]
                for a in policy[state]:
                    if a == max_action:
                        policy[state][a] = 1 - epsilon + (epsilon / len(policy[state]))
                    else:
                        policy[state][a] = epsilon / len(policy[state])

    return dict(policy), {k: dict(v) for k, v in Q.items()}



def off_policy_mc_control(env, num_episodes, gamma=0.99):
    """
    Implémente Off-Policy MC Control avec importance sampling.
    """
    Q = defaultdict(lambda: defaultdict(float))
    C = defaultdict(lambda: defaultdict(float))
    target_policy = {}
    behavior_policy = defaultdict(dict)
    
    for episode_num in range(num_episodes):
        episode = generate_episode(env, behavior_policy)
        G = 0.0
        W = 1.0

        for i in reversed(range(len(episode))):
            state, action, reward = episode[i]
            G = gamma * G + reward

            # Initialisation politique comportementale si besoin
            if state not in behavior_policy or not behavior_policy[state]:
                valid_actions = env.available_actions()
                prob = 1.0 / len(valid_actions)
                behavior_policy[state] = {a: prob for a in valid_actions}

            C[state][action] += W
            Q[state][action] += (W / C[state][action]) * (G - Q[state][action])

            # Mise à jour politique cible
            best_action = max(Q[state].items(), key=lambda x: x[1])[0]
            target_policy[state] = best_action

            if action != target_policy[state]:
                break
            
            # Importance sampling ratio
            W = W / behavior_policy[state][action]
            if W == 0:
                break

    return target_policy, {k: dict(v) for k, v in Q.items()}

In [ ]:
env = SecretEnv3()
pi_mc, Q_mc = on_policy_first_visit_mc_control(env, epsilon=0.1, num_episodes=1000)

# 3. Affichage de la politique extraite
print("\n Politique extraite (MC On-Policy, premiers états connus) :")
for state, actions in list(pi_mc.items())[:10]:
    best_action = max(actions, key=actions.get)
    print(f"État {state} → Action optimale : {best_action}")

# 4. Affichage des valeurs Q
print("\n Valeurs Q associées (premiers états connus) :")
for state, action_values in list(Q_mc.items())[:10]:
    print(f"État {state} → {action_values}")



🎯 Politique extraite (MC On-Policy, premiers états connus) :
État 51161 → Action optimale : 0
État 50906 → Action optimale : 0
État 50643 → Action optimale : 0
État 50387 → Action optimale : 1
État 50135 → Action optimale : 0
État 49878 → Action optimale : 0
État 49625 → Action optimale : 0
État 49370 → Action optimale : 0
État 49091 → Action optimale : 0
État 48835 → Action optimale : 1

📊 Valeurs Q associées (premiers états connus) :
État 51161 → {2: -0.14285714285714285, 0: -0.034482758620689655, 1: -0.5}
État 50906 → {0: -0.20973684210526317, 2: -0.3333333333333333}
État 50643 → {0: -0.26334444444444444}
État 50387 → {1: -0.4676808095238095}
État 50135 → {1: -0.5587182871428571, 0: -0.08018606019607843}
État 49878 → {2: -0.4136186177290323, 0: 0.21120032778888886}
État 49625 → {2: -0.4324908911401202, 0: 0.8966181761306923}
État 49370 → {0: -0.7002677497722843}
État 49091 → {0: -0.6277802348093067}
État 48835 → {1: -0.6215024324612135}


In [4]:
from collections import defaultdict
import random

def off_policy_mc_control(env, num_episodes, gamma=0.99):
    Q = defaultdict(lambda: defaultdict(float))
    C = defaultdict(lambda: defaultdict(float))
    target_policy = {}
    behavior_policy = defaultdict(dict)

    for episode_num in range(num_episodes):
        episode = generate_episode(env, behavior_policy)
        G = 0.0
        W = 1.0

        for i in reversed(range(len(episode))):
            state, action, reward = episode[i]
            G = gamma * G + reward

            # Récupère les actions valides dans cet état
            valid_actions = env.available_actions() if hasattr(env, "available_actions") else []

            # Initialise behavior_policy si nécessaire
            # Initialise behavior_policy si nécessaire
            if state not in behavior_policy or not behavior_policy[state]:
                if valid_actions is not None and len(valid_actions) > 0:
                    prob = 1.0 / len(valid_actions)
                    behavior_policy[state] = {a: prob for a in valid_actions}
                else:
                    continue  # Pas d’actions possibles dans cet état


            # Ignore cette transition si l'action est invalide dans la politique comportementale
            if action not in behavior_policy[state]:
                continue

            C[state][action] += W
            Q[state][action] += (W / C[state][action]) * (G - Q[state][action])

            best_action = max(Q[state].items(), key=lambda x: x[1])[0]
            target_policy[state] = best_action

            if action != target_policy[state]:
                break

            W = W / behavior_policy[state][action]
            if W == 0:
                break

    return target_policy, {k: dict(v) for k, v in Q.items()}


In [3]:
def off_policy_mc_control(env, num_episodes, gamma=0.99):
    Q = defaultdict(lambda: defaultdict(float))
    C = defaultdict(lambda: defaultdict(float))
    target_policy = {}
    behavior_policy = defaultdict(dict)

    def get_valid_actions(state):
        return env.available_actions() if hasattr(env, "available_actions") else []

    def init_behavior_policy(state, valid_actions):
        prob = 1.0 / len(valid_actions)
        return {a: prob for a in valid_actions}

    for _ in range(num_episodes):
        episode = generate_episode(env, behavior_policy)
        G = 0.0
        W = 1.0

        for i in reversed(range(len(episode))):
            state, action, reward = episode[i]
            G = gamma * G + reward

            if state not in behavior_policy or not behavior_policy[state]:
                valid_actions = get_valid_actions(state)
                if len(valid_actions) == 0:
                    continue
                behavior_policy[state] = init_behavior_policy(state, valid_actions)

            if action not in behavior_policy[state]:
                continue

            C[state][action] += W
            Q[state][action] += (W / C[state][action]) * (G - Q[state][action])

            best_action = max(Q[state].items(), key=lambda x: x[1])[0]
            target_policy[state] = best_action

            if action != target_policy[state]:
                break

            W = W / behavior_policy[state][action]
            if W == 0:
                break

    return target_policy, {k: dict(v) for k, v in Q.items()}


In [4]:
env = SecretEnv3()
target_policy, Q = off_policy_mc_control(env, num_episodes=1000)

#  Affiche la politique
print("\n Politique Off-Policy extraite (premiers états connus) :")
for i, (s, a) in enumerate(target_policy.items()):
    print(f"État {s} → Action optimale : {a}")
    if i >= 9:
        break

#  Affiche les valeurs Q associées
print("\n Valeurs Q associées (premiers états connus) :")
for i, (s, a_dict) in enumerate(Q.items()):
    print(f"État {s} → {a_dict}")
    if i >= 9:
        break



 Politique Off-Policy extraite (premiers états connus) :
État 50929 → Action optimale : 1
État 50681 → Action optimale : 1
État 49907 → Action optimale : 1
État 49655 → Action optimale : 1
État 49402 → Action optimale : 2
État 48890 → Action optimale : 2
État 48374 → Action optimale : 2
État 48113 → Action optimale : 2
État 47865 → Action optimale : 1
État 47349 → Action optimale : 2

 Valeurs Q associées (premiers états connus) :
État 50929 → {1: 0.0, 2: 0.0}
État 50681 → {1: 0.0, 2: -1.9801}
État 49907 → {1: 0.0}
État 49655 → {1: 0.0}
État 49402 → {2: -0.9642857142857143}
État 48890 → {2: 0.0}
État 48374 → {2: -0.9313286172692622}
État 48113 → {2: -1.99}
État 47865 → {1: -1.9701}
État 47349 → {2: -1.93089501}


In [6]:
from collections import defaultdict
import random

def mc_es_control(env, num_episodes, gamma=0.99):
    Q = defaultdict(lambda: defaultdict(float))
    returns = defaultdict(list)
    policy = defaultdict(lambda: defaultdict(float))

    for _ in range(num_episodes):
        # --- Exploring Start ---
        state = env.reset()
        state = env.state_id()
        valid_actions = env.available_actions()

        if valid_actions is None or len(valid_actions) == 0:
            continue

        action = random.choice(valid_actions)  # Action aléatoire initiale
        env.step(action)

        episode = []
        done = env.is_game_over()
        prev_score = env.score()
        episode.append((state, action, 0))  # Reward = 0 à l'init

        state = env.state_id()

        # --- Générer l’épisode jusqu’à la fin ---
        while not done:
            valid_actions = env.available_actions()
            if state in policy and policy[state]:
                actions = list(policy[state].keys())
                probs = list(policy[state].values())
                action = random.choices(actions, weights=probs, k=1)[0]
            else:
                action = random.choice(valid_actions)

            env.step(action)
            new_score = env.score()
            reward = new_score - prev_score
            prev_score = new_score

            episode.append((state, action, reward))
            state = env.state_id()
            done = env.is_game_over()

        # --- Mise à jour Q et politique ---
        G = 0
        visited = set()

        for t in reversed(range(len(episode))):
            s, a, r = episode[t]
            G = gamma * G + r

            if (s, a) not in visited:
                visited.add((s, a))
                returns[(s, a)].append(G)
                Q[s][a] = sum(returns[(s, a)]) / len(returns[(s, a)])

                best_action = max(Q[s].items(), key=lambda x: x[1])[0]
                valid_actions = Q[s].keys()
                for act in valid_actions:
                    policy[s][act] = 1.0 if act == best_action else 0.0

    return dict(policy), {k: dict(v) for k, v in Q.items()}


In [8]:
# ⚙️ 1. Instanciation de l’environnement
env = SecretEnv3()

#  2. Apprentissage avec MC Exploring Starts
policy_es, Q_es = mc_es_control(env, num_episodes=10000)

#  3. Affichage d’un extrait de la politique apprise
print("\nPolitique MC ES extraite (premiers états connus) :")
for i, (state, actions) in enumerate(policy_es.items()):
    best_action = max(actions.items(), key=lambda x: x[1])[0]
    print(f"État {state} → Action optimale : {best_action}")
    if i == 9:
        break

#  4. Affichage des valeurs Q associées
print("\nValeurs Q associées (premiers états connus) :")
for i, (state, actions) in enumerate(Q_es.items()):
    print(f"État {state} → {actions}")
    if i == 9:
        break



Politique MC ES extraite (premiers états connus) :
État 65093 → Action optimale : 0
État 64832 → Action optimale : 2
État 64576 → Action optimale : 1
État 64320 → Action optimale : 1
État 64200 → Action optimale : 1
État 63945 → Action optimale : 0
État 63690 → Action optimale : 0
État 63431 → Action optimale : 0
État 63174 → Action optimale : 2
État 62913 → Action optimale : 2

Valeurs Q associées (premiers états connus) :
État 65093 → {0: -0.05555555555555555}
État 64832 → {2: -0.10654761904761904}
État 64576 → {1: -0.18372042440318304}
État 64320 → {1: -0.24417988305489227}
État 64200 → {1: 0.7234907919249999}
État 63945 → {0: 0.59017069439083}
État 63690 → {0: 0.48081209298294786}
État 63431 → {0: 0.2826669653949919}
État 63174 → {2: 0.1100292811651863}
État 62913 → {2: -0.035011768970375265}
